<a href="https://colab.research.google.com/github/raven-in-space/data-science-cohort-20/blob/main/Project-2/Predicting_Housing_Prices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2: Housing Price Prediction


The project should be done using Regression prediction techniques ( with feature scaling and regularization ) as covered in class.
The goal is to minimize the RMS ***percentage*** error ( root mean squared percentage error - RMSPE ) on your prediction of the house sales price.



Any of the available features can be used in the regression, but a minimum set of variables that do the job should be the ultimate goal.



Be sure to go through the whole data science process and document as such in your Jupyter notebook.



This project will have less direct "To Do" guidance and the progression through the data science process will be more individualized this time around.
We will talk about all the issues during class so you're not going to be out on a ledge with this one, I just want to move you towards performing a data science project on your own eventually.



A data dictionary file is available at AWS S3 at [Housing Data Dictionary]( https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Housing%20-%20Data%20Documentation.pdf ).

The data is available on AWS S3 at https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv .


# Predicting Housing Prices Using Linear Regression and RMSPE
* **Date <u>Started</u>:** `4/10/2026`
* **Author:** Raven Otero-Symphony
* **Program:** CNM Ingenuity [Data Science Bootcamp]()

# Problem Definition

The Accessor's Office has provided data on individual residential properties sold from 2006 to 2010. This project aims to *reliably* predict house sales prices through feature scaling, regularization, and minimixing the **Root Mean Square *Percentage* Error (RMSPE)** in a predictive regression model. Secondarily, we aim to find the most reliable model with the *least* amount of variables possible to predict housing prices.

# Data Collection

We are using Python data cleaning and regression methods, including `sklearn`, to fit and evaluate our predictive model.

**Data Sources:**
1. [Source Data](https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv) - collected from AWS S3.
2. [Data Dictionary](https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Housing%20-%20Data%20Documentation.pdf) - a brief description of the data and its and features.

🎈 Stands for comments related to future development.

# Data Cleaning

## Load

In [1]:
import pandas as pd
import numpy as np

## IDE

In [2]:
orig_housing = pd.read_csv("https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv")
orig_housing.shape

(2637, 81)

In [3]:
orig_housing.head(5)

,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,905101070,20,RL,62.0,14299,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,7,2007,WD,Normal,115400
1,905101330,90,RL,72.0,10791,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,Shed,500,10,2006,WD,Normal,90000
2,903454090,50,RM,50.0,9000,Pave,NaN,Reg,Bnk,AllPub,...,0,NaN,NaN,NaN,0,12,2007,WD,Normal,141000
3,533244030,60,FV,68.0,7379,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,254000
4,909252020,70,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,4,2009,WD,Normal,155000


In [4]:
orig_housing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2637 entries, 0 to 2636
Data columns (total 81 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   PID              2637 non-null   int64  
 1   MS SubClass      2637 non-null   int64  
 2   MS Zoning        2637 non-null   object 
 3   Lot Frontage     2188 non-null   float64
 4   Lot Area         2637 non-null   int64  
 5   Street           2637 non-null   object 
 6   Alley            180 non-null    object 
 7   Lot Shape        2637 non-null   object 
 8   Land Contour     2637 non-null   object 
 9   Utilities        2637 non-null   object 
 10  Lot Config       2637 non-null   object 
 11  Land Slope       2637 non-null   object 
 12  Neighborhood     2637 non-null   object 
 13  Condition 1      2637 non-null   object 
 14  Condition 2      2637 non-null   object 
 15  Bldg Type        2637 non-null   object 
 16  House Style      2637 non-null   object 
 17  Overall Qual  

In [5]:
orig_housing['Garage Qual'].unique()

array(['TA', 'Fa', nan, 'Gd', 'Ex', 'Po'], dtype=object)

**Initial Observations:**
* According to the [data dictionary](https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Housing%20-%20Data%20Documentation.pdf), there should be **2,930 observations (rows)** and **82 variables (features)**. What we find instead is that there are **2,637 rows** and **81 columns**. That's a difference of **293 rows** and **1 column**. I wonder which feature has the most missing rows, and why?
* The `MS Subclass` feature is being picked up as an integer rather than the nominal variable it actually is. Type conversions will be necessary for some features!
* The classification of nominal versus ordinal variables is interesting within the dictionary. For example, `Lot Shape` is considered ordinal but `Land Contour` is considered nominal.
* The `Sawyer` versus `Sawyer West` naming in the `Neighborhood` feature is *hopefully* labeled correctly in the actual data... hopefully (re-classifying would require a judgement call since I can't actually reach out to the PoC).
* I might combine or drop `Condition 1` with `Condition 2`, `Exterior 1` with `Exterior 2`, `Exter Qual` with `Exter Cond`, `Bsmt Qual` with `Bsmt Cond`, depending on correlation strength and model relevancy, since the labels within these features repeat themselves.
* As an aside, it would be interesting to look at the `Overall Qual` and `Overall Cond` features (since those seem self-reported) against other features of the house. Did the surveyor assume quality in a consistent fashion across houses? Or did they hold some biases?
* `SalePrice` is the target variable and has no missing rows.
* Some nominal variables contain the value `NA` to signify the *lack* of the feature in question. However, this is being picked up as `NULL` in Python, therefore when reclassifying, these must be carefully treated. These include:
  * `Alley`
  * `Bsmt Cond`
  * `Bsmt Exposure`
  * `BsmtFin Type 1`
  * `BsmtFin Type 2`
  * `FireplaceQu`
  * `Garage Type`
  * `Garage Finish`
  * `Garage Qual`
  * `Garage Cond`
  * `Pool QC`
  * `Fence`

### Missing Data

In [6]:
# create a copy
main_housing = orig_housing.copy()

# Create a summary table of missing values
null_summary = pd.DataFrame({
    'Null Count': main_housing.isna().sum(),
    'Percentage': (main_housing.isna().sum() / len(main_housing)) * 100
}).sort_values(by='Null Count', ascending=False)

null_summary

,Null Count,Percentage
Pool QC,2626,99.582859
Misc Feature,2541,96.359499
Alley,2457,93.174061
Fence,2109,79.977247
Mas Vnr Type,1607,60.940463
...,...,...
Mo Sold,0,0.000000
Yr Sold,0,0.000000
Sale Type,0,0.000000
Sale Condition,0,0.000000


**Missing Data Summary:**
* **5 features** have **over 50%** of missing data. Although 48% of `Fireplace Qu` is missing, I'm going to drop that feature as well because it is so close to 50% in context of the rest of the data.
* **10 features** are missing **2-5%** of data and **1 feature** is missing **17%** of data. These can be imputed, although my hunch is that some of the features will be highly correlated amongst themselves (do data scientists work off of hunches? I do).
* **9 features** are missing less than **1%** of data. Again, this can be imputed if necessary.
* The remaining **57 features** aren't missing any data, including the target variable.

In [7]:
# copy
main_all_cols = main_housing.copy()

# store original columns
original_cols = set(main_housing.columns)

# drop features at cutoff
cutoff = ( len(main_housing) * 0.6 )
main_housing = main_housing.dropna(axis = 1, thresh=cutoff)

In [8]:
dropped_cols = original_cols - set(main_housing.columns)
print(f'The following columns are dropped for missing over 50% of data: {dropped_cols}')

The following columns are dropped for missing over 50% of data: {'Misc Feature', 'Fence', 'Pool QC', 'Fireplace Qu', 'Alley', 'Mas Vnr Type'}


In [9]:
# total NaNs across all rows
main_housing.isnull().sum().sum()

np.int64(1584)

* The [`set` method](https://www.geeksforgeeks.org/pandas/create-a-set-from-a-series-in-pandas/) is used to check which columns are not in the original data after being removed for meeting the `cutoff`.
* Strange how `Condition 2` appears more than `Condition 1` when the data dictionary states that `Condition 2` exists only if more than one characteristic from `Condition 1` exists...

### Variable Coercion

* To deal with the remaining missing values, we need to separate by either `numeric` from `categorical` data. **`Categorical` data will need to be converted into a `numeric` values in order to fit within a linear regression model.**
  * We will check the distributions of the `numeric` data before choosing either the `mean` or `median` to impute, if necessary.
  * If a `categorical` variable has only 2 outputs, we opt for **binary coercion**, since using one-hot encoding will result in multicollinearity (one value is always the inverse of the other).
  * If a `categorical` variable has more than 2 outputs, then we opt for **one-hot encoding** and check correlation.

In [10]:
# create filters for relevant summary stats
stats_cat = main_housing.select_dtypes(include='object') # "cat" stands for "categorical"
stats_num = main_housing.select_dtypes(include='number') # "num" stands for "numerical"

#### Categorical

* First, let's convert those `NA` values accidentally getting picked up as `NaN` into `None` for clarity.

In [11]:
# converting "NA" from NaN to readable values
# checking original values first
main_housing['Bsmt Cond'].unique()

array(['TA', nan, 'Gd', 'Fa', 'Po', 'Ex'], dtype=object)

In [12]:
# create list and replace NaN with "NA" in list
false_nans = ['Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond']

for col in false_nans:
  stats_cat[col] = ["None" if pd.isna(val) else val for val in main_housing[col]]

In [13]:
# verifying it worked
stats_cat['Bsmt Cond'].unique()

array(['TA', 'None', 'Gd', 'Fa', 'Po', 'Ex'], dtype=object)

* Now, let's visualize the trends in the categorical data:

In [14]:
# visualize the categorical features
stats_cat.describe().transpose().sort_values(by='freq', ascending=False)

,count,unique,top,freq
Utilities,2637,3,AllPub,2634
Street,2637,2,Pave,2625
Condition 2,2637,8,Norm,2609
Roof Matl,2637,8,CompShg,2599
Heating,2637,6,GasA,2597
Land Slope,2637,3,Gtl,2511
Central Air,2637,2,Y,2460
Functional,2637,8,Typ,2453
Electrical,2637,5,SBrkr,2414
Garage Cond,2637,6,TA,2400


In [15]:
# visualizing top freq
import plotly.express as px
fig = px.histogram(stats_cat, x='Utilities')
fig.show()

In [16]:
# visualizing lowest freq
import plotly.express as px
fig = px.histogram(stats_cat, x='Neighborhood')
fig.show()

In [17]:
# visualizing again to find binary v. 2+ values
stats_cat.describe().transpose().sort_values(by='unique', ascending=True)

,count,unique,top,freq
Street,2637,2,Pave,2625
Central Air,2637,2,Y,2460
Utilities,2637,3,AllPub,2634
Land Slope,2637,3,Gtl,2511
Paved Drive,2637,3,Y,2395
Lot Shape,2637,4,Reg,1662
Land Contour,2637,4,Lvl,2365
Exter Qual,2637,4,TA,1614
Garage Finish,2637,4,Unf,1106
Kitchen Qual,2637,4,TA,1342


In [18]:
# well isn't this a pain...
stats_cat['Paved Drive'].unique()

array(['N', 'Y', 'P'], dtype=object)

* `pandas` doesn't distinguish between **nominal** versus **ordinal** data, but the data dictionary *does.*
  * According to [Geeks for Geeks](https://www.geeksforgeeks.org/machine-learning/one-hot-encoding-vs-label-encoding/) (one of my favorite data science websites), one-hot encoding should be used for nominal data when the number of unique categories is relatively small.
  * Ordinal data requires much more of a judgement call since it assumes equal distance between data points, which may not always be true. (e.g., Is the difference between "Poor" and "Good" the same as "Great" and "Excellent"?)
* After checking the data dictionary, it looks like only `Paved Drive` doesn't technically reference `TRUE/FALSE` values. If it weren't for the third value, `P`, it could have been coerced into a `boolean` feature.
* Only `Street` and `Central Air` can be converted into `boolean` values. Every other `categorical` variable can be one-hot encoded.

In [19]:
# checking to ensure ordinal columns exist in data
stats_cat_cols = set(stats_cat.columns)

ordinal_cols = [
    'Lot Shape',
    'Utilities',
    'Land Slope',
    'Exter Qual',
    'Exter Cond',
    'Bsmt Qual',
    'Bsmt Cond',
    'Bsmt Exposure',
    'BsmtFin Type 1',
    'BsmtFin Type 2',
    'Electrical',
    'Functional',
    'Garage Finish',
    'Garage Qual',
    'Garage Cond',
    'Paved Drive'
]

if set(ordinal_cols) in stats_cat_cols:
  print("True")
else:
  print("False")

set(ordinal_cols) - stats_cat_cols

False


set()

In [20]:
# now checking nominal...
nominal_cols = [
    'PID',
    'MS SubClass',
    'MS Zoning',
    'Street',
    'Land Contour',
    'Lot Config',
    'Neighborhood',
    'Condition 1',
    'Condition 2',
    'Bldg Type',
    'House Style',
    'Roof Style',
    'Roof Matl',
    'Exterior 1st', # incorrectly named in dictionary
    'Exterior 2nd', # incorrectly named in dictionary
    'Foundation',
    'Heating',
    'Central Air',
    'Garage Type',
    'Sale Type',
    'Sale Condition'
]

if set(nominal_cols) in stats_cat_cols:
  print("True")
else:
  print("False")

set(nominal_cols) - stats_cat_cols

False


{'MS SubClass', 'PID'}

* Both `PID` and `MS SubClass` are incorrectly getting picked up as `numeric` types rather than as nominal data types.
  * We're ultimately dropping `PID` anyway because it serves as an identifier feature.
  * `MS SubClass` will get converted using label encoding because it is already ordered using numbers.

In [21]:
# drop PID
main_with_pid = main_housing.copy()

In [22]:
main_housing = main_housing.drop(axis=1, columns='PID')

In [23]:
# coerce MS SubClass
main_housing['MS SubClass'] = main_housing['MS SubClass'].astype(str)
main_housing['MS SubClass'].dtype

dtype('O')

In [24]:
main_housing['MS SubClass'].unique()

array(['20', '90', '50', '60', '70', '40', '120', '80', '160', '30',
       '190', '85', '180', '45', '75', '150'], dtype=object)

* Looking at nominal data first...

In [25]:
main_housing[ordinal_cols].describe().transpose().sort_values(by='unique',ascending=False)

,count,unique,top,freq
Functional,2637,8,Typ,2453
BsmtFin Type 1,2564,6,Unf,768
BsmtFin Type 2,2563,6,Unf,2245
Exter Cond,2637,5,TA,2290
Bsmt Qual,2564,5,TA,1161
Bsmt Cond,2564,5,TA,2358
Electrical,2637,5,SBrkr,2414
Garage Qual,2490,5,TA,2356
Garage Cond,2490,5,TA,2400
Lot Shape,2637,4,Reg,1662


In [26]:
main_housing['Functional'].unique()

array(['Typ', 'Min2', 'Maj1', 'Mod', 'Min1', 'Maj2', 'Sal', 'Sev'],
      dtype=object)

In [30]:
import plotly.express as px
fig = px.histogram(main_housing, x='Functional')
fig.show()

* I'm going to replace the "Minor" and "Major" values in `Functional` so that this feature has 6 unique values.

In [28]:
main_housing['Functional'] = main_housing['Functional'].replace(
    {'Min1': "Min",
     'Min2': "Min",
     'Maj1': "Maj",
     'Maj2': "Maj"})#.astype("int")
fig = px.histogram(main_housing, x='Functional')
fig.show()

* `Functional` is a very imbalanced feature, but we're ignoring it for now...

**🚩🚩🚩🚩🚩NEXT STEPS:**
* Judgement call on one-hot encoding ordinal data
* Investigating numeric data
* Checking correlations
* More drops
* Fit first linear regression model
* Stat summary to investigate best variables
* Repeat

#### Numeric

In [38]:
# now the numeric features
stats_num.describe().round(1).transpose().sort_values(by='count', ascending=False)

,count,mean,std,min,25%,50%,75%,max
PID,2637.0,714130147.7,188752674.8,526301100.0,528477010.0,535453040.0,907187010.0,1.007100e+09
MS SubClass,2637.0,57.3,42.5,20.0,20.0,50.0,70.0,1.900000e+02
Lot Area,2637.0,10044.7,6742.5,1300.0,7436.0,9450.0,11526.0,1.646600e+05
Overall Qual,2637.0,6.1,1.4,1.0,5.0,6.0,7.0,1.000000e+01
Overall Cond,2637.0,5.6,1.1,1.0,5.0,5.0,6.0,9.000000e+00
Year Built,2637.0,1971.3,30.3,1872.0,1954.0,1973.0,2001.0,2.010000e+03
Misc Val,2637.0,42.0,393.2,0.0,0.0,0.0,0.0,1.250000e+04
Year Remod/Add,2637.0,1984.2,20.9,1950.0,1965.0,1993.0,2004.0,2.010000e+03
1st Flr SF,2637.0,1155.5,382.6,334.0,878.0,1082.0,1380.0,4.692000e+03
Bedroom AbvGr,2637.0,2.9,0.8,0.0,2.0,3.0,3.0,6.000000e+00


---

In [25]:
stats_cat.columns

Index(['MS Zoning', 'Street', 'Alley', 'Lot Shape', 'Land Contour',
       'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl',
       'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC',
       'Central Air', 'Electrical', 'Kitchen Qual', 'Functional',
       'Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual',
       'Garage Cond', 'Paved Drive', 'Pool QC', 'Fence', 'Misc Feature',
       'Sale Type', 'Sale Condition'],
      dtype='object')

In [26]:
# Generate binary values using get_dummies
# dum_df = pd.get_dummies(foods_df, columns=["Food"] ) # Can change prefix using prefix argument
dum_df = pd.get_dummies(stats_cat, columns=['MS Zoning', 'Street', 'Alley', 'Lot Shape', 'Land Contour',
       'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl',
       'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC',
       'Central Air', 'Electrical', 'Kitchen Qual', 'Functional',
       'Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual',
       'Garage Cond', 'Paved Drive', 'Pool QC', 'Fence', 'Misc Feature',
       'Sale Type', 'Sale Condition'], prefix="", prefix_sep="") # Can change prefix using prefix argument
dum_df

,A (agr),C (all),FV,I (all),RH,RL,RM,Grvl,Pave,Grvl,...,New,Oth,VWD,WD,Abnorml,AdjLand,Alloca,Family,Normal,Partial
0,False,False,False,False,False,True,False,False,True,False,...,False,False,False,True,False,False,False,False,True,False
1,False,False,False,False,False,True,False,False,True,False,...,False,False,False,True,False,False,False,False,True,False
2,False,False,False,False,False,False,True,False,True,False,...,False,False,False,True,False,False,False,False,True,False
3,False,False,True,False,False,False,False,False,True,False,...,False,False,False,True,False,False,False,False,True,False
4,False,False,False,False,False,True,False,False,True,False,...,False,False,False,True,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,False,False,False,False,False,False,True,False,True,False,...,False,False,False,True,False,False,False,False,True,False
2633,False,False,False,False,False,True,False,False,True,False,...,False,False,False,True,False,False,False,False,True,False
2634,False,False,False,False,False,True,False,False,True,False,...,False,False,False,True,False,False,False,False,True,False
2635,False,False,False,False,False,True,False,False,True,False,...,False,False,False,True,False,False,False,False,True,False


In [27]:
import plotly.express as px

dum_correlation_matrix = dum_df.corr().round(2)
fig = px.imshow(dum_correlation_matrix)

fig.show()

# Exploratory Data Analysis

* Produce some visual analysis of the data – like plots showing the distributions of all variables. Recall that Gaussian Naive Bayes assumes the predictors are normally distributed. Note: you might have to do multiple plots in groups.

* NOTE: the ‘target’ column indicates a successful transaction (‘1’) or a no-transaction (‘0’). Verify these are the only values in that column.

* Check the correlation values between all predictor columns to ensure there are no substantial correlations between predictors. This is important to support the decision to classify the ‘target’ using Naïve Bayes.

* Create two data frames: one with all successful transactions, one with all unsuccessful transactions. Make sure they are copies and not slices.

In [23]:
import plotly.express as px

correlation_matrix = stats_num.corr().round(2)
fig = px.imshow(correlation_matrix)

fig.show()

# Data Processing

* Create two data frames: one with all the predictor columns (everything except for Unnamed: 0, ID_code and target) and one with just the target. Make sure they are copies and not slices.

* Define a Gaussian Naïve Bayes model using Sklearn.

* Divide the two data frames you created in step #10 into training and testing subsets.

* Train the model using the training subset of the dataset.

* Test the model using the testing subset of the dataset. Calculate and report the accuracy.

* Perform a cross-validation loop to calculate the accuracy of your model. Report that accuracy. How does it compare to the accuracy you calculated in #14?

* Plot a histogram of the accuracy scores you generated in your cross-validation loop. What do you notice about the distribution of accuracy scores?

* Present the confusion matrix and the results of your Classification Report (sklearn.metrics.classification_report). What do you notice?

* The training data is very skewed towards non-successful transactions (about 90% of the training data has ‘target’==0). Remove enough non-successful transaction rows so that your remaining training data is 50%/50% split between successful and non-successful transactions. Hint: you can use the data frames you created in step #9.

* Repeat the cross-validation process on this data set. Report what your cross-validation accuracy is in this 50/50 case.

# Data Visualization

* Compare the results of your cross-validation with the whole training data and the reduced 50/50 training data

* Present the confusion matrix and the results of your Classification Report (sklearn.metrics.classification_report)

# Communication of Results

* Communicate the results of your analysis.